In [1]:
from kingdon import Algebra
import sympy as sym
import numpy as np
import ipykernel
import ipywidgets
from math import cos, sin, pi, e, sqrt, acos, atan


ModuleNotFoundError: No module named 'sympy'

In [ ]:
# setting up the projective algebra PGA (3,0,1)
alg = Algebra(3, 0, 1)
locals().update(alg.blades)
point = lambda x, y, z: (e0 + x*e1 + y*e2 + z*e3).dual()
line = lambda x1, y1, z1:(e0 + x1*e12 + y1*e13 + z1*e23)
plane = lambda a, b, c, d: alg.vector(e1=a, e2=b, e3=c, e0=d)

# symbols for multivectors that represent axes and rotations
a1 = sym.symbols('a1', real=True)
a2 = sym.symbols('a2', real=True)
a3 = sym.symbols('a3', real=True)
ra1 = sym.symbols('ra1', real=True)
ra2 = sym.symbols('ra2', real=True)
a31 = sym.symbols('a31', real=True)
ra3 = sym.symbols('ra3', real=True)

# identificaction of axes to blades:
# e12: z-axis, e13: y-axis, e23: x-axis
a1 = e12
# a1 represents the z-axis and is encoded in the blade e12 of PGA (3,0,1)
# we need to construct the axes of the fundamental region in some orientation 
# we choose to use e12 (z-Axis upward) as one of the axes of the fundamental region in order to have a good orientation to keep track off.

# Rotor of 120° degrees around z-axis of the fundamental region of the symmetry group r332.
angle_120 = 4*sym.pi/3
ra1 = sym.cos(angle_120/2) +sym.sin(angle_120/2)*a1

# second axis of rotation of the fundamental region with angle (acos -1/3)/2 = -54°:
angle_1 = sym.acos(sym.Rational(-1,3))/2
r2 = sym.cos(angle_1/2) + sym.sin(angle_1/2) *e13
a2 =  r2 >> a1

# Rotor of 180° around the second axis of rotation of the fundamental region
ra2 = sym.cos(sym.pi/2)+ sym.sin(sym.pi/2)*a2

# third axis of rotation of the fundamental region with 2-fold symmetry : we apply a double rotation for the construction
# rotation around y-axis with acos(1/3) = 70.53°
angle_2 = sym.acos(sym.Rational(-1,3))
r3 = sym.cos(-angle_2/2) + sym.sin(-angle_2/2) *e13
r3
a31 = r3 >> a1    

# rotation around z-axis with 60°
angle_3 = -sym.pi/3
angle_3

r4 = sym.cos(angle_3/2) + sym.sin(angle_3/2) * e12
a3 = r4 >> a31

#  Rotor of 120° around the third axis of rotation of the fundamental region
ra3 = sym.cos(-sym.pi/3)+ sym.sin(-sym.pi/3)*a3

# origin
p0 = point (0,0,0)

# x-value of point P11 
x = sym.symbols('x', real=True, positive= True)

# y-value of point P11  
y = sym.symbols('y', real=True, positive= True)

# z-value of point P11  
z = sym.symbols('z', real=True, positive= True)

# Punkt 11 
p11 = point (x,y,z)

In [ ]:
# having defined the first point as a variable of its three coordinates x,y,z in E3 we construct all other points
# since all points are equal they can be constructed by using the rotors of the symmetry group
# this yields an algebraic constraint on all points defined as variables, 
# i.e. we get 12 different points in 3D but still have only 3 variables x,y and z to solve for

def points_tensegrity_tetrahedron (eckpunkt):

    p11 = eckpunkt    

    # Punkte von Ring 1
    p21 = ra3 >> p11
    p31 = ra3 >> p21

     # Punkte von Ring 2
    p12 = ra1 >> p11
    p22 = ra1 >> p21
    p32 = ra1 >> p31

    # Punkte von Ring 3
    p13 = ra1 >> p12
    p23 = ra1 >> p22
    p33 = ra1 >> p32

    # Punkte von Ring 4
    p43 = ra2 >> p12
    p41 = ra1 >> p43
    p42 = ra1 >> p41

    # from the vertices we can derive cables and plates and midpoints as multivectors handily with GA by simple addition:
    # for algebraic reasons we do not normalize the points yet, but leave that for later.
    # please refer for the naming convention to the labled images that show up further down. 
    
    # Center of the frist ring of three cables 
    r1m = p11 + p21 + p31
        
    # Center of plate 1
    q1m = p11 + p12 + p13

    # Center of plate 2
    q2m = p21 + p41 + p32

    # Center of plate 3
    q3m = p22 + p42 + p33
    
    # Center of plate 3
    q4m = p23 + p43 + p31

    return p11, p12, p13, p21, p22, p23, p31, p32, p33, p41, p42, p43, r1m, q1m, q2m, q3m, q4m

p11, p12, p13, p21, p22, p23, p31, p32, p33, p41, p42, p43, r1m, q1m, q2m, q3m, q4m = points_tensegrity_tetrahedron (p11)





In [ ]:
# equilibrium condition for p11: coplanarity of four points: p11, the center of an adjacent plate, center of the adjacent edge-cable and the center of the adjacent ringcables, which represents a resultant vector of the two ring-vectors at p11.
# by creating a regressive product of the four points we get the signed volume of the tetrahedron between four points
# if the points a coplanar the volume is zero. 
# So equating the tetrahedron spanned by the four points with zero is an algebraic expression for the coplanarity constraint.

eq1 = r1m & q1m & p23 & p11
eq1 = eq1.values()[0]
eq1 = eq1 *3/4

In [ ]:
# This coplanarity constraint leads to an cubic equation in x, y, z:
eq1

-sqrt(6)*x**3 - 3*sqrt(2)*x**2*y + sqrt(3)*x**2*z + 12*x*y*z + sqrt(6)*x*z**2 - 3*sqrt(3)*y**2*z + 3*sqrt(2)*y*z**2

In [ ]:
# Normalization of midpoints. In order to get cartesian distances of points in PGA, we need normalization.
r1m = r1m.normalized()
q1m = q1m.normalized()
q2m = q2m.normalized()
q3m = q3m.normalized()
q4m = q4m.normalized()

In [ ]:
# in order to knock out one variable or degree of freedom of the equation we can fix the radius of the plateb to 1
def dist_sq(a, b): 
    return alg.rp(a, b).normsq().e

eq2 = dist_sq(p11,q1m)-1
eq2 = sym.nsimplify(eq2)
y_solve = sym.solve(eq2,y)
eq1 = eq1.subs(y,y_solve[0])
eq1

-sqrt(6)*x**3 + sqrt(3)*x**2*z - 3*sqrt(2)*x**2*sqrt(1 - x**2) + sqrt(6)*x*z**2 + 12*x*z*sqrt(1 - x**2) + 3*sqrt(2)*z**2*sqrt(1 - x**2) - 3*sqrt(3)*z*(1 - x**2)

In [ ]:
# in order to knock out another variable we can set a rotation angle alpha of the plate
# this leads to some proportion of ring cables and edge cables representing a transformation from 
# a tetrahedron into a cube octahedron
alpha = sym.symbols('alpha', real=True)
x_value = sym.cos(alpha)
y_value = sym.sin(alpha)
eq1 = eq1.subs(x,x_value)
eq1 = eq1.subs(y,y_value)
eq1 = eq1.simplify()
eq1

sqrt(6)*z**2*cos(alpha) + 3*sqrt(2)*z**2*Abs(sin(alpha)) + 4*sqrt(3)*z*cos(alpha)**2 + 12*z*cos(alpha)*Abs(sin(alpha)) - 3*sqrt(3)*z - sqrt(6)*cos(alpha)**3 - 3*sqrt(2)*cos(alpha)**2*Abs(sin(alpha))

In [ ]:
# this yields a general algebraic expression for z with respect to a rotation-angle alpha.
z_solve = sym.solve(eq1,z)
z_solve[1]=z_solve[1].simplify()
z_solve[1]

(3*sqrt(-16*sin(alpha)**4 + 16*sqrt(3)*cos(alpha)**3*Abs(sin(alpha)) - 16*cos(alpha)**2 - 8*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 19) - 4*sqrt(3)*cos(alpha)**2 - 12*cos(alpha)*Abs(sin(alpha)) + 3*sqrt(3))/(2*(sqrt(6)*cos(alpha) + 3*sqrt(2)*Abs(sin(alpha))))

In [ ]:
# we can calculate algebraically the proportion of ring-cables and edge-cables, but the expression is not nice: 

x_alg = x_value
y_alg = y_value
z_alg = z_solve[1]

# this leads to an algebraic expression for p11 and all other points: 
p11 = point(x_alg,y_alg,z_alg)
p11, p12, p13, p21, p22, p23, p31, p32, p33, p41, p42, p43, r1m, q1m, q2m, q3m, q4m = points_tensegrity_tetrahedron (p11)

In [ ]:
p12



((-3*sqrt(-16*sin(alpha)**4 + 16*sqrt(3)*cos(alpha)**3*Abs(sin(alpha)) - 16*cos(alpha)**2 - 8*sqrt(3)*cos(alpha)*Abs(sin(alpha)) + 19) + 4*sqrt(3)*cos(alpha)**2 + 12*cos(alpha)*Abs(sin(alpha)) - 3*sqrt(3))/(2*(sqrt(6)*cos(alpha) + 3*sqrt(2)*Abs(sin(alpha))))) 𝐞₀₁₂ + (cos(alpha + pi/6)) 𝐞₀₁₃ + (sin(alpha + pi/6)) 𝐞₀₂₃ + (1) 𝐞₁₂₃

In [ ]:
# now we derive the proportion of the cables by looking at the distance of their endpoints:
cable_ring = p11 & p12
dist_sq1 = sum(c**2 for c in cable_ring.values())
cable_edge = p11 & p23
dist_sq2 = sum(c**2 for c in cable_edge.values())


In [ ]:
# proportion of edge-cable-length squared and ring-cable-length squared

prop_cable_sq = dist_sq2 / dist_sq1
prop_cable_sq = sym.simplify(prop_cable_sq)
print(prop_cable_sq)


4*(sqrt(6)*cos(alpha) + 3*sqrt(2)*Abs(sin(alpha)))**2*(16*(sqrt(3)*cos(alpha) + 3*Abs(sin(alpha)))**2*(12*sqrt(6)*cos(alpha)**3 - 15*sqrt(6)*cos(alpha) + sqrt(6)*cos(3*alpha) - 18*sqrt(2)*Abs(sin(alpha)))**2*sin(alpha)**2 + 4*(sqrt(3)*cos(alpha) + 3*Abs(sin(alpha)))**2*(3*sqrt(-16*sin(alpha)**4 + 4*sqrt(3)*cos(alpha)*Abs(sin(alpha)) - 8*cos(2*alpha) + 4*sqrt(3)*cos(3*alpha)*Abs(sin(alpha)) + 11)*cos(alpha) + 6*sqrt(3)*sqrt(-16*sin(alpha)**4 + 4*sqrt(3)*cos(alpha)*Abs(sin(alpha)) - 8*cos(2*alpha) + 4*sqrt(3)*cos(3*alpha)*Abs(sin(alpha)) + 11)*cos(2*alpha)*Abs(sin(alpha)) - 3*sqrt(-16*sin(alpha)**4 + 4*sqrt(3)*cos(alpha)*Abs(sin(alpha)) - 8*cos(2*alpha) + 4*sqrt(3)*cos(3*alpha)*Abs(sin(alpha)) + 11)*cos(3*alpha) + 16*sqrt(3)*sin(alpha)**4*cos(alpha) - sqrt(3)*cos(alpha) + 6*cos(2*alpha)*Abs(sin(alpha)) - 2*sqrt(3)*cos(3*alpha) + 3*sqrt(3)*cos(5*alpha) - 12*Abs(sin(alpha)))**2 + (12*sqrt(6)*cos(alpha)**3 - 15*sqrt(6)*cos(alpha) + sqrt(6)*cos(3*alpha) - 18*sqrt(2)*Abs(sin(alpha)))**2*((sqr

In [ ]:
# with this algebraic expression we may specify a cable proportion in order to get a rotation angle 
eq3 = prop_cable_sq - 9
alpha_solve = sym.nonlinsolve([eq3],[alpha]) 
alpha_solve



In [ ]:
# now we may chose a rotation angle alpha to get real valued solutions to all three variables:
alpha_num = np.pi/6
x = sym.cos(alpha_num)
y = sym.sin(alpha_num)
z = z_solve[1].subs(alpha, alpha_num).evalf()


In [ ]:
z

In [ ]:
# now that we solved all variables, we can visualize the solution:

p11 = point(x,y,z)
p11, p12, p13, p21, p22, p23, p31, p32, p33, p41, p42, p43, r1m, q1m, q2m, q3m, q4m = points_tensegrity_tetrahedron (p11)

In [ ]:
# we can calculate numerically the proportion of ring-cables and edge-cables:
length_cable_ring = dist_sq(p11,p12)
length_cable_edge = dist_sq(p11,p23)
prop_cable = (length_cable_ring) / (length_cable_edge)
prop_cable.evalf()


In [ ]:
# Linien bzw. Seile von Ring 1
s11 = [p11.map(float), p21.map(float)]
s12 = [p21.map(float), p31.map(float)]
s13 = [p31.map(float), p11.map(float)]

# Linien bzw. Seile von Ring 2
s21 = [p12.map(float), p22.map(float)]
s22 = [p22.map(float), p32.map(float)]
s23 = [p32.map(float), p12.map(float)]

# Linien bzw. Seile von Ring 3
s31 = [p13.map(float), p23.map(float)]
s32 = [p23.map(float), p33.map(float)]
s33 = [p33.map(float), p13.map(float)]

# Linien bzw. Seile von Ring 4
s41 = [p41.map(float), p42.map(float)]
s42 = [p42.map(float), p43.map(float)]
s43 = [p43.map(float), p41.map(float)]

# Kantenseile
ks1 = [p11.map(float), p23.map(float)]
ks2 = [p21.map(float), p12.map(float)]
ks3 = [p31.map(float), p41.map(float)]
ks4 = [p22.map(float), p13.map(float)]
ks5 = [p33.map(float), p43.map(float)]
ks6 = [p42.map(float), p32.map(float)]

# Platten
# Platte 1
f1 = [[p11.map(float), p12.map(float), p13.map(float)],[p11.map(float),p12.map(float)], [p12.map(float),p13.map(float)], [p13.map(float),p11.map(float)]]

# Platte 2
f2 = [[p21.map(float), p41.map(float), p32.map(float)],[p21.map(float),p41.map(float)], [p41.map(float),p32.map(float)], [p32.map(float),p21.map(float)]]

# Platte 3
f3 = [[p22.map(float), p42.map(float), p33.map(float)],[p22.map(float),p42.map(float)], [p42.map(float),p33.map(float)], [p33.map(float),p22.map(float)]]

# Platte 4
f4 = [[p23.map(float), p43.map(float), p31.map(float)],[p23.map(float),p43.map(float)], [p43.map(float),p31.map(float)], [p31.map(float),p23.map(float)]]

In [ ]:
# Create the colored tripod list for ganja.js
# Format: [hex_color, point1, point2, "label"]

x_point = p0 + 1.0 * e23  # Move along X
y_point = p0 + 1.0 * e13  # Move along Y (note: sign depends on orientation)
z_point = p0 + 1.0 * e12  # Move along Z

axes = [
    0xff0000, p0, x_point, "X-Achse",  # Red for X
    0x00ff00, p0, y_point, "Y-Achse",  # Green for Y
    0x0000ff, p0, z_point, "Z-Achse",  # Blue for Z

]
def graph_func():
    return [
        "Tensegrity Tetrahedron",
        *axes,
        0x550000, p0.map(float), "Ursprung",
        0x550000, a1.map(float), "a1",
        0x550000, a2.map(float), "a2",
        0x550000, a3.map(float), "a3",
        
        0x555500, p11.map(float), "p11",
        0x555500, p12.map(float), "p12",
        0x555500, p13.map(float), "p13",
        
        0x555500, p21.map(float), "p21",
        0x555500, p22.map(float), "p22",
        0x555500, p23.map(float), "p23",

        0x555500, p31.map(float), "p31",
        0x555500, p32.map(float), "p32",
        0x555500, p33.map(float), "p33", 

        0x555500, p41.map(float), "p41",
        0x555500, p42.map(float), "p42",
        0x555500, p43.map(float), "p43",

        0x555500, q1m.map(float), "q1m",
        0x555500, q2m.map(float), "q2m",
        0x555500, q3m.map(float), "q3m",
        0x555500, q4m.map(float), "q4m",
 
        0x555500, s11, "s11",
        0x555500, s12, "s12",
        0x555500, s13, "s13",

        0x555500, s21, "s21",
        0x555500, s22, "s22",
        0x555500, s23, "s23",

        0x555500, s31, "s31",
        0x555500, s32, "s32",
        0x555500, s33, "s33",

        0x555500, s41, "s41",
        0x555500, s42, "s42",
        0x555500, s43, "s43", 

        0x555500, ks1, "ks1",
        0x555500, ks2, "ks2",
        0x555500, ks3, "ks3",
        0x555500, ks4, "ks4",
        0x555500, ks5, "ks5",
        0x555500, ks6, "ks6",
        
        0x555500, f1[1],f1[2], f1[3],
        0x555500, f2[1],f2[2], f2[3],
        0x555500, f3[1],f3[2], f3[3],
        0x555500, f4[1],f4[2], f4[3],
        
        '<G opacity="0.4">', 
        0x222222,f1[0],
        0x222222,f2[0],
        0x222222,f3[0],
        0x222222,f4[0],
       '</G>',        
    ]

alg.graph( 
    graph_func,
    lineWidth=1,
    grid= False, 
    labels=False,
    pointRadius=0.8,
    gl=0,
    fontSize=0.8,
    scale=0.8,
    height='800px'
)